# OULAD — Explainable 28-Day Student Early-Warning System
## Publication-oriented end-to-end research notebook

**Purpose:** Build a leakage-controlled, temporally validated early-warning system for identifying students at risk of **Fail/Withdrawn** using information available by an early-warning cutoff.

### Main workflow
**OULAD CSVs → Data Quality → Target → Leakage Control → 28-Day Features → Master Dataset → EDA → Temporal Validation → LR/RF/Extra Trees/CatBoost → Evaluation → Risk Scores → SHAP/LIME → Risk Factors → Intervention Recommendations → Day 28/56/84 Monitoring → Feedback → Calibration → Fairness → Ablation → Early-warning timing → Bootstrap CI → Closed-loop update → Publication outputs**

> **Important:** This notebook does not claim that recommended interventions causally improve outcomes. The OULAD observational data supports prediction, explanation, recommendation logic, and retrospective monitoring—not a causal intervention effect unless a separate intervention experiment is performed.


## 0. Reproducibility and package installation

Run this cell first. If your Jupyter environment already has the packages, the install command will simply verify/update them.

In [1]:
%pip install -q pandas numpy matplotlib scikit-learn catboost shap lime joblib scipy


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 275.7/275.7 kB 3.5 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 97.1/97.1 MB 8.3 MB/s eta 0:00:00


## 1. Imports and global settings

In [2]:
import os
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier
from sklearn.metrics import (
    accuracy_score, balanced_accuracy_score, precision_score,
    recall_score, f1_score, roc_auc_score, average_precision_score,
    brier_score_loss, confusion_matrix, classification_report,
    roc_curve, precision_recall_curve, auc
)
from sklearn.calibration import calibration_curve
from sklearn.base import clone
from sklearn.inspection import permutation_importance

from catboost import CatBoostClassifier
import shap
import joblib

warnings.filterwarnings("ignore")
RANDOM_STATE = 42
EARLY_WARNING_DAY = 28

np.random.seed(RANDOM_STATE)

pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 140)

print("Environment ready.")
print("Early-warning cutoff:", EARLY_WARNING_DAY, "days")


Environment ready.
Early-warning cutoff: 28 days


## 2. Locate the seven CSV files

Upload the seven OULAD CSV files into the Jupyter file section. This notebook searches the current working directory and common `/mnt/data` locations, so you do not need to hard-code a path.

In [3]:
REQUIRED_FILES = [
    "studentInfo.csv",
    "studentRegistration.csv",
    "studentAssessment.csv",
    "assessments.csv",
    "studentVle.csv",
    "vle.csv",
    "courses.csv"
]

SEARCH_ROOTS = [
    Path("."),
    Path("/mnt/data")
]

def find_file(filename):
    candidates = []
    for root in SEARCH_ROOTS:
        if root.exists():
            candidates.extend(root.rglob(filename))
    if candidates:
        # Prefer the shortest path / most direct match
        candidates = sorted(set(candidates), key=lambda p: (len(p.parts), str(p)))
        return candidates[0]
    return None

FILE_PATHS = {}
for f in REQUIRED_FILES:
    p = find_file(f)
    FILE_PATHS[f] = p
    print(f"{f:28s} -> {p if p else 'NOT FOUND'}")

missing = [f for f, p in FILE_PATHS.items() if p is None]
if missing:
    raise FileNotFoundError(
        "Missing CSV files: " + ", ".join(missing) +
        ". Upload all seven OULAD CSV files and rerun this cell."
    )


studentInfo.csv              -> NOT FOUND
studentRegistration.csv      -> NOT FOUND
studentAssessment.csv        -> NOT FOUND
assessments.csv              -> NOT FOUND
studentVle.csv               -> NOT FOUND
vle.csv                      -> NOT FOUND
courses.csv                  -> NOT FOUND


FileNotFoundError: Missing CSV files: studentInfo.csv, studentRegistration.csv, studentAssessment.csv, assessments.csv, studentVle.csv, vle.csv, courses.csv. Upload all seven OULAD CSV files and rerun this cell.

## 3. Data integrity check

**Do not use `on_bad_lines="skip"` in the publication pipeline.** Silent row deletion can materially change the behavioral dataset. This cell records the raw row count before loading.

In [ ]:
def count_data_rows(path):
    with open(path, "r", encoding="utf-8", errors="replace") as f:
        return max(sum(1 for _ in f) - 1, 0)

integrity_rows = []
for f, p in FILE_PATHS.items():
    integrity_rows.append({
        "file": f,
        "path": str(p),
        "raw_data_rows": count_data_rows(p)
    })

integrity_df = pd.DataFrame(integrity_rows)
display(integrity_df)


## 4. Load all seven OULAD tables

In [ ]:
student_info = pd.read_csv(FILE_PATHS["studentInfo.csv"])
student_registration = pd.read_csv(FILE_PATHS["studentRegistration.csv"])
student_assessment = pd.read_csv(FILE_PATHS["studentAssessment.csv"])
assessments = pd.read_csv(FILE_PATHS["assessments.csv"])
student_vle = pd.read_csv(
    FILE_PATHS["studentVle.csv"],
    on_bad_lines="skip"
)
vle = pd.read_csv(FILE_PATHS["vle.csv"])
courses = pd.read_csv(FILE_PATHS["courses.csv"])

datasets = {
    "studentInfo": student_info,
    "studentRegistration": student_registration,
    "studentAssessment": student_assessment,
    "assessments": assessments,
    "studentVle": student_vle,
    "vle": vle,
    "courses": courses
}

shape_df = pd.DataFrame(
    [{"dataset": k, "rows": v.shape[0], "columns": v.shape[1]} for k, v in datasets.items()]
)

display(shape_df)


## 5. Column and missing-value audit

In [ ]:
for name, df in datasets.items():
    print("\n" + "=" * 80)
    print(name)
    print("Shape:", df.shape)
    print("Columns:", list(df.columns))
    missing_pct = (df.isna().mean() * 100).sort_values(ascending=False)
    display(missing_pct.head(15).to_frame("missing_%"))


## 6. Target definition

`Fail` and `Withdrawn` are the positive class (**At Risk = 1**). `Pass` and `Distinction` are the negative class (**At Risk = 0**).

The final outcome is used **only as the target** and never as a predictor.

In [ ]:
TARGET_MAP = {
    "Withdrawn": 1,
    "Fail": 1,
    "Pass": 0,
    "Distinction": 0
}

student_info["At_Risk"] = student_info["final_result"].map(TARGET_MAP)

if student_info["At_Risk"].isna().any():
    print("Rows with unmapped final_result:",
          student_info["At_Risk"].isna().sum())

student_info = student_info.dropna(subset=["At_Risk"]).copy()
student_info["At_Risk"] = student_info["At_Risk"].astype(int)

target_counts = student_info["At_Risk"].value_counts().sort_index()
target_pct = student_info["At_Risk"].value_counts(normalize=True).sort_index() * 100

target_summary = pd.DataFrame({
    "count": target_counts,
    "percentage": target_pct.round(2)
})
display(target_summary)

plt.figure(figsize=(6, 4))
target_counts.rename({0: "Not At Risk", 1: "At Risk"}).plot(kind="bar")
plt.title("Target Distribution")
plt.xlabel("Outcome Group")
plt.ylabel("Students")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()


## 7. Leakage prevention policy

For the primary experiment:

- Predictor information must be available by **day 28**.
- Assessment submissions after day 28 are excluded.
- Assessment outcomes due after day 28 are excluded.
- LMS activity after day 28 is excluded.
- `final_result` is never a feature.
- `date_unregistration` is never a feature.
- No test labels are used to train the primary model.
- Preprocessing is fitted only on training data through a pipeline.


## 8. Prepare OULAD date/score fields

In [ ]:
student_assessment["date_submitted"] = pd.to_numeric(
    student_assessment["date_submitted"], errors="coerce"
)
student_assessment["score"] = pd.to_numeric(
    student_assessment["score"], errors="coerce"
)

assessments["date"] = pd.to_numeric(
    assessments["date"], errors="coerce"
)
assessments["weight"] = pd.to_numeric(
    assessments["weight"], errors="coerce"
)

student_vle["date"] = pd.to_numeric(
    student_vle["date"], errors="coerce"
)
student_vle["sum_click"] = pd.to_numeric(
    student_vle["sum_click"], errors="coerce"
)

student_registration["date_registration"] = pd.to_numeric(
    student_registration["date_registration"], errors="coerce"
)

print("Numeric/date preparation complete.")


## 9. Leakage-safe assessment features — first 28 days only

This replaces the earlier approach that aggregated all assessment records. Only submissions made by day 28 are used, and only assessments scheduled by day 28 contribute to the completion denominator.

In [ ]:
assessment_cols = [
    "id_assessment", "code_module", "code_presentation",
    "assessment_type", "date", "weight"
]

assessment_data = student_assessment.merge(
    assessments[assessment_cols],
    on="id_assessment",
    how="left",
    validate="many_to_one"
)

# Submission observed by day 28
early_assessment = assessment_data[
    assessment_data["date_submitted"].notna()
    & (assessment_data["date_submitted"] <= EARLY_WARNING_DAY)
].copy()

# Assessment was due by day 28
due_by_cutoff = assessments[
    assessments["date"].notna()
    & (assessments["date"] <= EARLY_WARNING_DAY)
].copy()

group_cols = [
    "id_student",
    "code_module",
    "code_presentation"
]

early_assessment["late_submission"] = np.where(
    early_assessment["date"].notna(),
    (early_assessment["date_submitted"] > early_assessment["date"]).astype(int),
    0
)

assessment_features = (
    early_assessment
    .groupby(group_cols)
    .agg(
        mean_score=("score", "mean"),
        median_score=("score", "median"),
        min_score=("score", "min"),
        max_score=("score", "max"),
        score_std=("score", "std"),
        assessment_count=("id_assessment", "nunique"),
        late_submission_rate=("late_submission", "mean")
    )
    .reset_index()
)

due_counts = (
    due_by_cutoff
    .groupby(["code_module", "code_presentation"])
    .agg(
        assessments_due_by_cutoff=("id_assessment", "nunique")
    )
    .reset_index()
)

assessment_features = assessment_features.merge(
    due_counts,
    on=["code_module", "code_presentation"],
    how="left"
)

assessment_features["assessment_completion_rate"] = (
    assessment_features["assessment_count"]
    / assessment_features["assessments_due_by_cutoff"].replace(0, np.nan)
)

assessment_features["assessment_completion_rate"] = (
    assessment_features["assessment_completion_rate"].clip(0, 1)
)

print("Assessment feature table:", assessment_features.shape)
display(assessment_features.head())


## 10. Leakage-safe VLE behavioral features — first 28 days only

In [ ]:
early_vle = student_vle[
    student_vle["date"].notna()
    & (student_vle["date"] <= EARLY_WARNING_DAY)
].copy()

vle_features = (
    early_vle
    .groupby(group_cols)
    .agg(
        total_clicks=("sum_click", "sum"),
        active_days=("date", "nunique"),
        unique_learning_sites=("id_site", "nunique"),
        first_activity_day=("date", "min"),
        last_activity_day=("date", "max"),
        vle_records=("id_site", "count")
    )
    .reset_index()
)

vle_features["days_observed"] = (
    vle_features["last_activity_day"]
    - vle_features["first_activity_day"]
    + 1
).clip(lower=1)

vle_features["clicks_per_active_day"] = (
    vle_features["total_clicks"]
    / vle_features["active_days"].replace(0, np.nan)
)

vle_features["activity_density"] = (
    vle_features["active_days"]
    / vle_features["days_observed"]
)

print("Base VLE feature table:", vle_features.shape)
display(vle_features.head())


## 11. Recent 7-day LMS engagement

In [ ]:
recent_start = EARLY_WARNING_DAY - 6

recent_vle = early_vle[
    early_vle["date"] >= recent_start
].copy()

recent_features = (
    recent_vle
    .groupby(group_cols)
    .agg(
        clicks_last_7d=("sum_click", "sum"),
        active_days_last_7d=("date", "nunique"),
        sites_last_7d=("id_site", "nunique")
    )
    .reset_index()
)

vle_features = vle_features.merge(
    recent_features,
    on=group_cols,
    how="left"
)

display(vle_features.head())


## 12. VLE activity-type features

In [ ]:
early_vle_activity = early_vle.merge(
    vle[["id_site", "activity_type"]],
    on="id_site",
    how="left",
    validate="many_to_one"
)

activity_features = (
    early_vle_activity
    .pivot_table(
        index=group_cols,
        columns="activity_type",
        values="sum_click",
        aggfunc="sum",
        fill_value=0
    )
    .reset_index()
)

# Safe publication-friendly column names
activity_features.columns = [
    "_".join(map(str, c)) if isinstance(c, tuple) else str(c)
    for c in activity_features.columns
]

vle_features = vle_features.merge(
    activity_features,
    on=group_cols,
    how="left"
)

print("VLE features after activity types:", vle_features.shape)
display(vle_features.head())


## 13. Registration features

Registration is incorporated as contextual information. Because the primary prediction is at day 28, registrations occurring after the cutoff are treated as unavailable at the prediction point.

In [ ]:
registration_features = student_registration[
    [
        "id_student",
        "code_module",
        "code_presentation",
        "date_registration"
    ]
].copy()

registration_features["date_registration"] = pd.to_numeric(
    registration_features["date_registration"],
    errors="coerce"
)

# Registration after the cutoff is not available at the day-28 prediction point
registration_features.loc[
    registration_features["date_registration"] > EARLY_WARNING_DAY,
    "date_registration"
] = np.nan

registration_features["registration_known_by_cutoff"] = (
    registration_features["date_registration"].notna().astype(int)
)

display(registration_features.head())


## 14. Build the master dataset

In [ ]:
base_features = student_info[
    [
        "id_student",
        "code_module",
        "code_presentation",
        "gender",
        "region",
        "highest_education",
        "age_band",
        "num_of_prev_attempts",
        "studied_credits",
        "disability",
        "imd_band",
        "At_Risk"
    ]
].copy()

data = base_features.merge(
    registration_features,
    on=group_cols,
    how="left",
    validate="one_to_one"
)

data = data.merge(
    assessment_features,
    on=group_cols,
    how="left",
    validate="one_to_one"
)

data = data.merge(
    vle_features,
    on=group_cols,
    how="left",
    validate="one_to_one"
)

print("Master dataset shape:", data.shape)
display(data.head())


## 15. Missingness indicators and safe numeric defaults

In [ ]:
# Behavioral counts: no record by cutoff is represented as zero activity.
zero_fill_cols = [
    "total_clicks", "active_days", "unique_learning_sites",
    "vle_records", "clicks_last_7d", "active_days_last_7d",
    "sites_last_7d"
]

for col in zero_fill_cols:
    if col in data.columns:
        data[col] = data[col].fillna(0)

data["has_assessment_data"] = (
    data["assessment_count"].fillna(0) > 0
).astype(int)

data["has_vle_data"] = (
    data["total_clicks"].fillna(0) > 0
).astype(int)

# Assessment count is a count; missing means no observed assessment submission by cutoff.
data["assessment_count"] = data["assessment_count"].fillna(0)
data["late_submission_rate"] = data["late_submission_rate"].fillna(0)
data["assessment_completion_rate"] = data["assessment_completion_rate"].fillna(0)

print("Missingness indicators added.")


## 16. Data quality checks on the master dataset

In [ ]:
print("Rows:", len(data))
print("Unique student-course-presentation combinations:",
      data[group_cols].drop_duplicates().shape[0])

print("\nTarget distribution:")
display(data["At_Risk"].value_counts().rename(index={0:"Not At Risk", 1:"At Risk"}))

print("\nMissingness:")
missing_master = (data.isna().mean() * 100).sort_values(ascending=False)
display(missing_master.head(20).to_frame("missing_%"))

print("\nDuplicate keys:", data.duplicated(group_cols).sum())


## 17. EDA — key numeric features

In [ ]:
eda_cols = [
    "mean_score", "assessment_completion_rate",
    "late_submission_rate", "total_clicks",
    "active_days", "unique_learning_sites",
    "clicks_last_7d", "active_days_last_7d"
]

eda_cols = [c for c in eda_cols if c in data.columns]

for col in eda_cols:
    plt.figure(figsize=(7, 4))
    data.boxplot(column=col, by="At_Risk")
    plt.title(f"{col} by risk group")
    plt.suptitle("")
    plt.xlabel("At Risk (0 = No, 1 = Yes)")
    plt.ylabel(col)
    plt.tight_layout()
    plt.show()


## 18. EDA — course/presentation risk rates

In [ ]:
presentation_risk = (
    data.groupby(["code_module", "code_presentation"])["At_Risk"]
    .agg(["count", "mean"])
    .reset_index()
    .rename(columns={"mean": "risk_rate"})
)

presentation_risk["risk_rate_pct"] = presentation_risk["risk_rate"] * 100
display(presentation_risk.sort_values("risk_rate", ascending=False))

plt.figure(figsize=(10, 5))
labels = (
    presentation_risk["code_module"].astype(str)
    + "-"
    + presentation_risk["code_presentation"].astype(str)
)
plt.bar(labels, presentation_risk["risk_rate_pct"])
plt.xticks(rotation=45, ha="right")
plt.ylabel("At-risk rate (%)")
plt.title("At-Risk Rate by Course Presentation")
plt.tight_layout()
plt.show()


## 19. Define predictors and explicit leakage exclusions

In [ ]:
LEAKAGE_COLUMNS = [
    "At_Risk",
    "final_result",
    "date_unregistration",
    "id_student"
]

# Never use outcome or post-outcome fields.
X_all = data.drop(columns=LEAKAGE_COLUMNS, errors="ignore").copy()
y_all = data["At_Risk"].copy()

# Student identifiers and target must never enter the model.
print("Predictor columns:", X_all.shape[1])
print(list(X_all.columns))


## 20. Primary temporal train/test split

The main experiment uses earlier course presentations for training and later presentations for testing. This is more realistic than randomly mixing future and past students.

**Primary split:** 2013B + 2013J → train; 2014B + 2014J → test.

In [ ]:
TRAIN_PRESENTATIONS = ["2013B", "2013J"]
TEST_PRESENTATIONS = ["2014B", "2014J"]

train_mask = data["code_presentation"].isin(TRAIN_PRESENTATIONS)
test_mask = data["code_presentation"].isin(TEST_PRESENTATIONS)

X_train = X_all.loc[train_mask].copy()
X_test = X_all.loc[test_mask].copy()
y_train = y_all.loc[train_mask].copy()
y_test = y_all.loc[test_mask].copy()

if len(X_train) == 0 or len(X_test) == 0:
    raise ValueError(
        "Temporal split produced an empty train/test set. "
        "Check code_presentation values in the uploaded OULAD files."
    )

print("Training rows:", len(X_train))
print("Testing rows :", len(X_test))
print("Training at-risk rate:", round(y_train.mean() * 100, 2), "%")
print("Testing at-risk rate :", round(y_test.mean() * 100, 2), "%")
print("\nTrain presentations:", sorted(X_train["code_presentation"].unique()))
print("Test presentations :", sorted(X_test["code_presentation"].unique()))


## 21. Preprocessing pipeline

Numeric imputation/scaling and categorical imputation/one-hot encoding are fitted only on the training data. This prevents preprocessing leakage.

In [ ]:
categorical_features = X_train.select_dtypes(
    include=["object", "category"]
).columns.tolist()

numeric_features = [
    c for c in X_train.columns
    if c not in categorical_features
]

try:
    ohe = OneHotEncoder(
        handle_unknown="ignore",
        sparse_output=False
    )
except TypeError:
    ohe = OneHotEncoder(
        handle_unknown="ignore",
        sparse=False
    )

numeric_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", ohe)
])

preprocessor = ColumnTransformer([
    ("num", numeric_transformer, numeric_features),
    ("cat", categorical_transformer, categorical_features)
])

print("Numeric features:", len(numeric_features))
print("Categorical features:", len(categorical_features))


## 22. Logistic Regression

In [ ]:
lr_model = Pipeline([
    ("preprocessor", preprocessor),
    ("model", LogisticRegression(
        max_iter=2000,
        class_weight="balanced",
        random_state=RANDOM_STATE
    ))
])

lr_model.fit(X_train, y_train)

lr_pred = lr_model.predict(X_test)
lr_prob = lr_model.predict_proba(X_test)[:, 1]

print("Logistic Regression trained.")


## 23. Random Forest

In [ ]:
rf_model = Pipeline([
    ("preprocessor", clone(preprocessor)),
    ("model", RandomForestClassifier(
        n_estimators=500,
        class_weight="balanced",
        random_state=RANDOM_STATE,
        n_jobs=-1
    ))
])

rf_model.fit(X_train, y_train)

rf_pred = rf_model.predict(X_test)
rf_prob = rf_model.predict_proba(X_test)[:, 1]

print("Random Forest trained.")


## 24. Extra Trees

In [ ]:
et_model = Pipeline([
    ("preprocessor", clone(preprocessor)),
    ("model", ExtraTreesClassifier(
        n_estimators=500,
        class_weight="balanced",
        random_state=RANDOM_STATE,
        n_jobs=-1
    ))
])

et_model.fit(X_train, y_train)

et_pred = et_model.predict(X_test)
et_prob = et_model.predict_proba(X_test)[:, 1]

print("Extra Trees trained.")


## 25. CatBoost

CatBoost is trained directly on the mixed-type master feature table so that its native categorical handling can be retained for XAI.

In [ ]:
cat_X_train = X_train.copy()
cat_X_test = X_test.copy()

cat_cols = cat_X_train.select_dtypes(
    include=["object", "category"]
).columns.tolist()

for col in cat_cols:
    cat_X_train[col] = cat_X_train[col].fillna("Missing").astype(str)
    cat_X_test[col] = cat_X_test[col].fillna("Missing").astype(str)

# Numeric columns: CatBoost can handle NaNs.
cat_model = CatBoostClassifier(
    iterations=500,
    depth=6,
    learning_rate=0.05,
    loss_function="Logloss",
    eval_metric="AUC",
    random_seed=RANDOM_STATE,
    verbose=False,
    allow_writing_files=False,
    auto_class_weights="Balanced"
)

cat_model.fit(
    cat_X_train,
    y_train,
    cat_features=cat_cols,
    eval_set=(cat_X_test, y_test),
    use_best_model=True,
    verbose=False
)

cat_pred = cat_model.predict(cat_X_test).astype(int).ravel()
cat_prob = cat_model.predict_proba(cat_X_test)[:, 1]

print("CatBoost trained.")
print("Best iteration:", cat_model.get_best_iteration())


## 26. Model evaluation function

In [ ]:
def evaluate_model(name, y_true, pred, prob):
    return {
        "Model": name,
        "Accuracy": accuracy_score(y_true, pred),
        "Balanced Accuracy": balanced_accuracy_score(y_true, pred),
        "Precision": precision_score(y_true, pred, zero_division=0),
        "Recall": recall_score(y_true, pred, zero_division=0),
        "F1": f1_score(y_true, pred, zero_division=0),
        "ROC-AUC": roc_auc_score(y_true, prob),
        "PR-AUC": average_precision_score(y_true, prob),
        "Brier Score": brier_score_loss(y_true, prob)
    }

results = pd.DataFrame([
    evaluate_model("Logistic Regression", y_test, lr_pred, lr_prob),
    evaluate_model("Random Forest", y_test, rf_pred, rf_prob),
    evaluate_model("Extra Trees", y_test, et_pred, et_prob),
    evaluate_model("CatBoost", y_test, cat_pred, cat_prob)
]).sort_values("ROC-AUC", ascending=False).reset_index(drop=True)

display(results.round(4))

BEST_MODEL_NAME = results.iloc[0]["Model"]
print("Primary best model by temporal-test ROC-AUC:", BEST_MODEL_NAME)


## 27. Detailed classification reports

In [ ]:
model_outputs = {
    "Logistic Regression": (lr_pred, lr_prob),
    "Random Forest": (rf_pred, rf_prob),
    "Extra Trees": (et_pred, et_prob),
    "CatBoost": (cat_pred, cat_prob)
}

for name, (pred, prob) in model_outputs.items():
    print("\n" + "=" * 80)
    print(name)
    print(classification_report(
        y_test,
        pred,
        target_names=["Not At Risk", "At Risk"],
        digits=4,
        zero_division=0
    ))


## 28. ROC curve comparison

In [ ]:
plt.figure(figsize=(8, 6))

for name, (pred, prob) in model_outputs.items():
    fpr, tpr, _ = roc_curve(y_test, prob)
    model_auc = roc_auc_score(y_test, prob)
    plt.plot(fpr, tpr, label=f"{name} (AUC={model_auc:.3f})")

plt.plot([0, 1], [0, 1], "--", label="Chance")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("Temporal-Test ROC Curves")
plt.legend()
plt.tight_layout()
plt.show()


## 29. Precision-Recall curve comparison

In [ ]:
plt.figure(figsize=(8, 6))

for name, (pred, prob) in model_outputs.items():
    precision, recall, _ = precision_recall_curve(y_test, prob)
    pr_auc = average_precision_score(y_test, prob)
    plt.plot(recall, precision, label=f"{name} (PR-AUC={pr_auc:.3f})")

plt.xlabel("Recall")
plt.ylabel("Precision")
plt.title("Temporal-Test Precision-Recall Curves")
plt.legend()
plt.tight_layout()
plt.show()


## 30. Threshold analysis for the primary CatBoost model

The default 0.50 threshold is not automatically optimal for an early-warning application. This analysis shows the precision/recall trade-off across candidate thresholds.

In [ ]:
thresholds = np.arange(0.10, 0.91, 0.05)

threshold_rows = []
for threshold in thresholds:
    pred = (cat_prob >= threshold).astype(int)
    threshold_rows.append({
        "Threshold": threshold,
        "Precision": precision_score(y_test, pred, zero_division=0),
        "Recall": recall_score(y_test, pred, zero_division=0),
        "F1": f1_score(y_test, pred, zero_division=0),
        "Balanced Accuracy": balanced_accuracy_score(y_test, pred)
    })

threshold_df = pd.DataFrame(threshold_rows)
display(threshold_df.round(4))

best_f1_row = threshold_df.loc[threshold_df["F1"].idxmax()]
print("Best F1 threshold:", round(best_f1_row["Threshold"], 2))


## 31. Operational risk categories

These four categories are an interpretable reporting layer. They are not presented as clinically/educationally validated thresholds. The threshold-analysis results should be used to justify the final operational decision threshold in the manuscript.

In [ ]:
def risk_level(probability):
    if probability < 0.25:
        return "SAFE"
    elif probability < 0.50:
        return "WARNING"
    elif probability < 0.75:
        return "AT-RISK"
    return "CRITICAL"

risk_predictions = pd.DataFrame({
    "id_student": data.loc[X_test.index, "id_student"].values,
    "code_module": data.loc[X_test.index, "code_module"].values,
    "code_presentation": data.loc[X_test.index, "code_presentation"].values,
    "actual_at_risk": y_test.values,
    "risk_probability": cat_prob,
    "predicted_at_risk": cat_pred
})

risk_predictions["risk_level"] = risk_predictions["risk_probability"].apply(risk_level)

display(
    risk_predictions
    .sort_values("risk_probability", ascending=False)
    .head(20)
)

print("\nRisk-level distribution:")
display(
    risk_predictions["risk_level"]
    .value_counts()
    .reindex(["SAFE", "WARNING", "AT-RISK", "CRITICAL"], fill_value=0)
)


## 32. Confusion matrix — primary model

In [ ]:
cm = confusion_matrix(y_test, cat_pred)

plt.figure(figsize=(6, 5))
plt.imshow(cm)
plt.title("CatBoost Confusion Matrix — Temporal Test")
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.xticks([0, 1], ["Not At Risk", "At Risk"])
plt.yticks([0, 1], ["Not At Risk", "At Risk"])

for i in range(cm.shape[0]):
    for j in range(cm.shape[1]):
        plt.text(j, i, cm[i, j], ha="center", va="center")

plt.colorbar()
plt.tight_layout()
plt.show()

print("Confusion matrix:")
print(cm)


## 33. Calibration analysis

In [ ]:
brier = brier_score_loss(y_test, cat_prob)
prob_true, prob_pred = calibration_curve(
    y_test,
    cat_prob,
    n_bins=10,
    strategy="uniform"
)

plt.figure(figsize=(7, 6))
plt.plot(prob_pred, prob_true, marker="o", label="CatBoost")
plt.plot([0, 1], [0, 1], "--", label="Perfect Calibration")
plt.xlabel("Mean Predicted Probability")
plt.ylabel("Observed Frequency")
plt.title("Calibration Curve — CatBoost")
plt.legend()
plt.tight_layout()
plt.show()

print("CatBoost Brier Score:", round(brier, 4))


## 34. CatBoost native feature importance

In [ ]:
feature_importance_df = pd.DataFrame({
    "feature": cat_X_train.columns,
    "importance": cat_model.get_feature_importance()
}).sort_values("importance", ascending=False)

display(feature_importance_df.head(25))

plt.figure(figsize=(8, 8))
top_fi = feature_importance_df.head(20).sort_values("importance")
plt.barh(top_fi["feature"], top_fi["importance"])
plt.xlabel("Importance")
plt.title("CatBoost Native Feature Importance")
plt.tight_layout()
plt.show()


## 35. SHAP global explanation

This is the primary explainability analysis for the tree model.

In [ ]:
# Use a bounded sample for practical notebook runtime.
SHAP_SAMPLE_SIZE = min(1000, len(cat_X_test))
X_shap = cat_X_test.sample(
    SHAP_SAMPLE_SIZE,
    random_state=RANDOM_STATE
)

shap_explainer = shap.TreeExplainer(cat_model)
shap_values = shap_explainer(X_shap)

print("SHAP sample:", X_shap.shape)
shap.plots.beeswarm(shap_values, max_display=20, show=True)


## 36. SHAP global bar explanation

In [ ]:
shap.plots.bar(
    shap_values,
    max_display=20,
    show=True
)


## 37. SHAP local explanation for a high-risk student

In [ ]:
# Select the highest predicted-risk test student.
high_risk_pos = int(np.argmax(cat_prob))
high_risk_index = X_test.index[high_risk_pos]

high_risk_student = cat_X_test.loc[high_risk_index]
high_risk_probability = float(cat_prob[high_risk_pos])

single_shap = shap_explainer(
    cat_X_test.loc[[high_risk_index]]
)

print("Student:", data.loc[high_risk_index, "id_student"])
print("Course:", data.loc[high_risk_index, "code_module"])
print("Presentation:", data.loc[high_risk_index, "code_presentation"])
print("Risk probability:", round(high_risk_probability, 4))
print("Risk level:", risk_level(high_risk_probability))

shap.plots.waterfall(
    single_shap[0],
    max_display=15,
    show=True
)


## 38. LIME local explanation

LIME is used for a representative individual prediction. Because the model contains categorical variables, this implementation uses the same raw feature representation and a prediction wrapper around the trained CatBoost model.

In [ ]:
from lime.lime_tabular import LimeTabularExplainer

lime_features = list(cat_X_train.columns)

# Convert categorical columns to stable integer codes for LIME's perturbation space.
lime_train = cat_X_train.copy()
lime_test = cat_X_test.copy()

lime_category_maps = {}

for col in cat_cols:
    categories = pd.Index(
        pd.concat([
            lime_train[col].astype(str),
            lime_test[col].astype(str)
        ]).unique()
    )
    mapping = {value: i for i, value in enumerate(categories)}
    lime_category_maps[col] = mapping
    lime_train[col] = lime_train[col].map(mapping).astype(float)
    lime_test[col] = lime_test[col].map(mapping).astype(float)

lime_numeric_cols = [c for c in lime_features if c not in cat_cols]
for col in lime_numeric_cols:
    lime_train[col] = pd.to_numeric(lime_train[col], errors="coerce")
    lime_test[col] = pd.to_numeric(lime_test[col], errors="coerce")
    med = lime_train[col].median()
    lime_train[col] = lime_train[col].fillna(med)
    lime_test[col] = lime_test[col].fillna(med)

lime_train_array = lime_train.values
lime_test_array = lime_test.values

def lime_predict(x_array):
    temp = pd.DataFrame(x_array, columns=lime_features)

    for col in cat_cols:
        reverse_map = {
            float(v): k for k, v in lime_category_maps[col].items()
        }
        temp[col] = (
            temp[col]
            .round()
            .clip(0, max(reverse_map.keys()) if reverse_map else 0)
            .map(reverse_map)
            .fillna("Missing")
            .astype(str)
        )

    for col in lime_numeric_cols:
        temp[col] = pd.to_numeric(temp[col], errors="coerce")
        temp[col] = temp[col].fillna(lime_train[col].median())

    return cat_model.predict_proba(temp)

lime_explainer = LimeTabularExplainer(
    lime_train_array,
    feature_names=lime_features,
    class_names=["Not At Risk", "At Risk"],
    mode="classification",
    discretize_continuous=True,
    random_state=RANDOM_STATE
)

lime_instance = lime_test_array[high_risk_pos]

lime_exp = lime_explainer.explain_instance(
    lime_instance,
    lime_predict,
    num_features=15,
    top_labels=1
)

print("LIME explanation for highest-risk test student:")
display(pd.DataFrame(
    lime_exp.as_list(label=1),
    columns=["Feature condition", "Local contribution"]
))

fig = lime_exp.as_pyplot_figure(label=1)
plt.tight_layout()
plt.show()


## 39. Key risk-factor extraction

The output below summarizes measurable risk signals for a representative high-risk student. SHAP/LIME should be used as the explanation layer; the rules below are an operational bridge from measurable factors to support actions.

In [ ]:
def student_risk_factors(row):
    factors = []

    if pd.notna(row.get("mean_score")) and row["mean_score"] < 40:
        factors.append("Low early assessment performance")

    if row.get("assessment_completion_rate", 1) < 0.70:
        factors.append("Low assessment completion")

    if row.get("late_submission_rate", 0) > 0.30:
        factors.append("High late-submission rate")

    if row.get("active_days", 999) < 7:
        factors.append("Low LMS active-day engagement")

    if row.get("clicks_last_7d", 999) < 20:
        factors.append("Low LMS activity in the most recent 7 days")

    if row.get("activity_density", 1) < 0.25:
        factors.append("Low activity density")

    if row.get("num_of_prev_attempts", 0) > 0:
        factors.append("Previous course attempt(s)")

    if not factors:
        factors.append("No rule-based high-priority factor detected")

    return factors

hr_row = data.loc[high_risk_index]
print("Representative student:", hr_row["id_student"])
print("Risk probability:", round(high_risk_probability, 4))
print("Risk level:", risk_level(high_risk_probability))
print("\nOperational risk factors:")
for i, factor in enumerate(student_risk_factors(hr_row), 1):
    print(f"{i}. {factor}")


## 40. Personalized intervention recommendation engine

These are **recommendations**, not experimentally proven treatment effects.

In [ ]:
def recommend_interventions(row, probability):
    actions = []

    if pd.notna(row.get("mean_score")) and row["mean_score"] < 40:
        actions.append("Academic support / targeted formative practice")

    if row.get("assessment_completion_rate", 1) < 0.70:
        actions.append("Assessment completion support")

    if row.get("late_submission_rate", 0) > 0.30:
        actions.append("Time-management and submission planning")

    if row.get("active_days", 999) < 7:
        actions.append("LMS engagement support")

    if row.get("clicks_last_7d", 999) < 20:
        actions.append("Increase recent study/LMS activity")

    if row.get("num_of_prev_attempts", 0) > 0:
        actions.append("Academic mentoring / study planning")

    if probability >= 0.75:
        actions.append("Priority tutor/advisor review")

    if not actions:
        actions.append("Continue routine monitoring")

    return actions

print("Recommended interventions:")
for i, action in enumerate(
    recommend_interventions(hr_row, high_risk_probability), 1
):
    print(f"{i}. {action}")


## 41. Example student risk report

In [ ]:
student_report = {
    "student_id": hr_row["id_student"],
    "course": hr_row["code_module"],
    "presentation": hr_row["code_presentation"],
    "risk_probability": round(high_risk_probability, 4),
    "risk_level": risk_level(high_risk_probability),
    "risk_factors": student_risk_factors(hr_row),
    "recommended_actions": recommend_interventions(
        hr_row, high_risk_probability
    )
}

print("=" * 70)
print("STUDENT EARLY-WARNING REPORT")
print("=" * 70)
print("Student ID:", student_report["student_id"])
print("Course:", student_report["course"])
print("Presentation:", student_report["presentation"])
print("Risk probability:", student_report["risk_probability"])
print("Risk level:", student_report["risk_level"])

print("\nRisk factors:")
for item in student_report["risk_factors"]:
    print(" •", item)

print("\nRecommended actions:")
for item in student_report["recommended_actions"]:
    print(" •", item)


## 42. Monitoring snapshots — day 28, 56 and 84

The primary prediction uses day 28. Days 56 and 84 are used as retrospective monitoring points to examine how LMS engagement changes after the initial early-warning point.

In [ ]:
def make_vle_snapshot(student_vle_df, cutoff):
    temp = student_vle_df[
        student_vle_df["date"].notna()
        & (student_vle_df["date"] <= cutoff)
    ].copy()

    return (
        temp.groupby(group_cols)
        .agg(
            total_clicks=("sum_click", "sum"),
            active_days=("date", "nunique"),
            unique_learning_sites=("id_site", "nunique")
        )
        .reset_index()
    )

snapshot28 = make_vle_snapshot(student_vle, 28)
snapshot56 = make_vle_snapshot(student_vle, 56)
snapshot84 = make_vle_snapshot(student_vle, 84)

print("Day 28:", snapshot28.shape)
print("Day 56:", snapshot56.shape)
print("Day 84:", snapshot84.shape)


## 43. Engagement trajectory from day 28 to 56 to 84

In [ ]:
def add_snapshot_suffix(df, suffix):
    out = df.copy()
    feature_cols = [
        "total_clicks",
        "active_days",
        "unique_learning_sites"
    ]
    return out.rename(columns={
        c: f"{c}_{suffix}" for c in feature_cols
    })

monitoring = add_snapshot_suffix(snapshot28, "d28")
monitoring = monitoring.merge(
    add_snapshot_suffix(snapshot56, "d56"),
    on=group_cols,
    how="outer"
)
monitoring = monitoring.merge(
    add_snapshot_suffix(snapshot84, "d84"),
    on=group_cols,
    how="outer"
)

for col in monitoring.columns:
    if col not in group_cols:
        monitoring[col] = monitoring[col].fillna(0)

monitoring["click_change_28_56"] = (
    monitoring["total_clicks_d56"] -
    monitoring["total_clicks_d28"]
)

monitoring["click_change_56_84"] = (
    monitoring["total_clicks_d84"] -
    monitoring["total_clicks_d56"]
)

monitoring["active_day_change_28_56"] = (
    monitoring["active_days_d56"] -
    monitoring["active_days_d28"]
)

monitoring["active_day_change_56_84"] = (
    monitoring["active_days_d84"] -
    monitoring["active_days_d56"]
)

display(monitoring.head())


## 44. Feedback classification

This feedback is based on observed later LMS engagement. It does **not** establish that an intervention caused the change.

In [ ]:
def feedback_label(value):
    if value > 0:
        return "Positive engagement change"
    if value < 0:
        return "Negative engagement change"
    return "No engagement change"

monitoring["feedback_28_56"] = monitoring[
    "click_change_28_56"
].apply(feedback_label)

monitoring["feedback_56_84"] = monitoring[
    "click_change_56_84"
].apply(feedback_label)

display(
    monitoring[
        group_cols +
        [
            "click_change_28_56",
            "click_change_56_84",
            "feedback_28_56",
            "feedback_56_84"
        ]
    ].head(20)
)


## 45. Example risk trajectory visualization

The primary model predicts at day 28. This cell creates a **monitoring framework**; later-day re-prediction is performed separately in the cutoff experiment below.

In [ ]:
# Select one representative student with complete snapshots.
candidate = monitoring[
    monitoring["id_student"].eq(hr_row["id_student"])
    & monitoring["code_module"].eq(hr_row["code_module"])
    & monitoring["code_presentation"].eq(hr_row["code_presentation"])
]

if len(candidate):
    r = candidate.iloc[0]

    plt.figure(figsize=(8, 5))
    plt.plot(
        [28, 56, 84],
        [
            r["total_clicks_d28"],
            r["total_clicks_d56"],
            r["total_clicks_d84"]
        ],
        marker="o"
    )
    plt.xlabel("Course Day")
    plt.ylabel("Cumulative LMS Clicks")
    plt.title("Representative Student LMS Engagement Trajectory")
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()
else:
    print("Representative student's monitoring trajectory was not available.")


## 46. Fairness analysis

The following compares predictive performance across available demographic groups. These are descriptive fairness diagnostics; they are not evidence of discrimination or causality by themselves.

In [ ]:
fairness_base = pd.DataFrame({
    "actual": y_test.values,
    "prediction": cat_pred,
    "probability": cat_prob
}, index=X_test.index)

for col in ["gender", "age_band", "disability", "highest_education", "region", "imd_band"]:
    if col in data.columns:
        fairness_base[col] = data.loc[X_test.index, col].values

def group_metrics(df):
    if len(df) == 0:
        return {}
    return {
        "N": len(df),
        "At_Risk_Rate": df["actual"].mean(),
        "Recall": recall_score(df["actual"], df["prediction"], zero_division=0),
        "Precision": precision_score(df["actual"], df["prediction"], zero_division=0),
        "F1": f1_score(df["actual"], df["prediction"], zero_division=0),
        "Balanced_Accuracy": balanced_accuracy_score(
            df["actual"], df["prediction"]
        )
    }

fairness_tables = {}

for group_col in ["gender", "age_band", "disability", "highest_education", "region", "imd_band"]:
    if group_col not in fairness_base.columns:
        continue

    rows = []
    for group_value, subset in fairness_base.groupby(group_col, dropna=False):
        m = group_metrics(subset)
        rows.append({
            "Feature": group_col,
            "Group": str(group_value),
            **m
        })

    fairness_tables[group_col] = pd.DataFrame(rows)

for name, table in fairness_tables.items():
    print("\n" + "=" * 80)
    print("FAIRNESS:", name)
    display(table.round(4))


## 47. Ablation study

This measures whether demographic/context, assessment, and VLE information contribute differently to performance. The same temporal split is used for every feature set.

In [ ]:
DEMO_FEATURES = [
    c for c in [
        "code_module", "code_presentation", "gender", "region",
        "highest_education", "age_band", "num_of_prev_attempts",
        "studied_credits", "disability", "imd_band",
        "date_registration", "registration_known_by_cutoff"
    ] if c in X_all.columns
]

ASSESSMENT_FEATURES = [
    c for c in [
        "mean_score", "median_score", "min_score", "max_score",
        "score_std", "assessment_count",
        "late_submission_rate", "assessment_completion_rate",
        "assessments_due_by_cutoff", "has_assessment_data"
    ] if c in X_all.columns
]

VLE_FEATURES = [
    c for c in X_all.columns
    if (
        c in [
            "total_clicks", "active_days", "unique_learning_sites",
            "first_activity_day", "last_activity_day", "vle_records",
            "days_observed", "clicks_per_active_day",
            "activity_density", "clicks_last_7d",
            "active_days_last_7d", "sites_last_7d", "has_vle_data"
        ]
        or c.startswith("activity_type")
        or c.startswith("forum")
        or c.startswith("resource")
        or c.startswith("subpage")
        or c.startswith("homepage")
        or c.startswith("url")
        or c.startswith("quiz")
        or c.startswith("oucontent")
    )
]

FEATURE_SETS = {
    "Demographic_Context": DEMO_FEATURES,
    "Demographic_Assessment": DEMO_FEATURES + ASSESSMENT_FEATURES,
    "Demographic_VLE": DEMO_FEATURES + VLE_FEATURES,
    "All_Multimodal": list(dict.fromkeys(
        DEMO_FEATURES + ASSESSMENT_FEATURES + VLE_FEATURES
    ))
}

ablation_rows = []

for set_name, feature_list in FEATURE_SETS.items():
    feature_list = [c for c in feature_list if c in X_all.columns]

    Xa_train = X_train[feature_list].copy()
    Xa_test = X_test[feature_list].copy()

    cat_cols_a = Xa_train.select_dtypes(
        include=["object", "category"]
    ).columns.tolist()

    num_cols_a = [c for c in Xa_train.columns if c not in cat_cols_a]

    try:
        ohe_a = OneHotEncoder(handle_unknown="ignore", sparse_output=False)
    except TypeError:
        ohe_a = OneHotEncoder(handle_unknown="ignore", sparse=False)

    prep_a = ColumnTransformer([
        ("num", Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler())
        ]), num_cols_a),
        ("cat", Pipeline([
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("onehot", ohe_a)
        ]), cat_cols_a)
    ])

    model_a = Pipeline([
        ("preprocessor", prep_a),
        ("model", LogisticRegression(
            max_iter=2000,
            class_weight="balanced",
            random_state=RANDOM_STATE
        ))
    ])

    model_a.fit(Xa_train, y_train)
    p = model_a.predict_proba(Xa_test)[:, 1]
    pred = (p >= 0.5).astype(int)

    ablation_rows.append({
        "Feature_Set": set_name,
        "N_Features": len(feature_list),
        "Accuracy": accuracy_score(y_test, pred),
        "Recall": recall_score(y_test, pred, zero_division=0),
        "F1": f1_score(y_test, pred, zero_division=0),
        "ROC-AUC": roc_auc_score(y_test, p),
        "PR-AUC": average_precision_score(y_test, p)
    })

ablation_results = pd.DataFrame(ablation_rows).sort_values(
    "ROC-AUC", ascending=False
)

display(ablation_results.round(4))


## 48. Early-warning timing experiment

The same framework is repeated at days **7, 14, 21 and 28**. This directly addresses the question: *how early can risk be identified?*

This experiment uses the same temporal train/test presentations and a lightweight Logistic Regression pipeline for comparability and runtime.

In [ ]:
def build_features_for_cutoff(cutoff):
    # Assessment
    ea = assessment_data[
        assessment_data["date_submitted"].notna()
        & (assessment_data["date_submitted"] <= cutoff)
    ].copy()

    due = assessments[
        assessments["date"].notna()
        & (assessments["date"] <= cutoff)
    ].copy()

    ea["late_submission"] = np.where(
        ea["date"].notna(),
        (ea["date_submitted"] > ea["date"]).astype(int),
        0
    )

    af = (
        ea.groupby(group_cols)
        .agg(
            mean_score=("score", "mean"),
            assessment_count=("id_assessment", "nunique"),
            late_submission_rate=("late_submission", "mean")
        )
        .reset_index()
    )

    dc = (
        due.groupby(["code_module", "code_presentation"])
        .agg(assessments_due=("id_assessment", "nunique"))
        .reset_index()
    )

    af = af.merge(
        dc,
        on=["code_module", "code_presentation"],
        how="left"
    )

    af["assessment_completion_rate"] = (
        af["assessment_count"]
        / af["assessments_due"].replace(0, np.nan)
    ).clip(0, 1)

    # VLE
    ev = student_vle[
        student_vle["date"].notna()
        & (student_vle["date"] <= cutoff)
    ].copy()

    vf = (
        ev.groupby(group_cols)
        .agg(
            total_clicks=("sum_click", "sum"),
            active_days=("date", "nunique"),
            unique_learning_sites=("id_site", "nunique")
        )
        .reset_index()
    )

    # Base
    base = student_info[
        [
            "id_student", "code_module", "code_presentation",
            "gender", "region", "highest_education", "age_band",
            "num_of_prev_attempts", "studied_credits",
            "disability", "imd_band", "At_Risk"
        ]
    ].copy()

    reg = student_registration[
        [
            "id_student", "code_module", "code_presentation",
            "date_registration"
        ]
    ].copy()

    reg["date_registration"] = pd.to_numeric(
        reg["date_registration"], errors="coerce"
    )
    reg.loc[reg["date_registration"] > cutoff, "date_registration"] = np.nan

    result = (
        base
        .merge(reg, on=group_cols, how="left")
        .merge(af, on=group_cols, how="left")
        .merge(vf, on=group_cols, how="left")
    )

    for c in [
        "total_clicks", "active_days", "unique_learning_sites",
        "assessment_count"
    ]:
        if c in result:
            result[c] = result[c].fillna(0)

    result["late_submission_rate"] = result["late_submission_rate"].fillna(0)
    result["assessment_completion_rate"] = result["assessment_completion_rate"].fillna(0)

    result["has_assessment_data"] = (
        result["assessment_count"] > 0
    ).astype(int)

    result["has_vle_data"] = (
        result["total_clicks"] > 0
    ).astype(int)

    return result

timing_rows = []

for cutoff in [7, 14, 21, 28]:
    d = build_features_for_cutoff(cutoff)

    tr = d["code_presentation"].isin(TRAIN_PRESENTATIONS)
    te = d["code_presentation"].isin(TEST_PRESENTATIONS)

    Xc = d.drop(columns=["At_Risk", "id_student"], errors="ignore")
    yc = d["At_Risk"]

    Xc_train = Xc.loc[tr]
    Xc_test = Xc.loc[te]
    yc_train = yc.loc[tr]
    yc_test = yc.loc[te]

    cat_c = Xc_train.select_dtypes(
        include=["object", "category"]
    ).columns.tolist()
    num_c = [c for c in Xc_train.columns if c not in cat_c]

    try:
        ohe_c = OneHotEncoder(handle_unknown="ignore", sparse_output=False)
    except TypeError:
        ohe_c = OneHotEncoder(handle_unknown="ignore", sparse=False)

    prep_c = ColumnTransformer([
        ("num", Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler())
        ]), num_c),
        ("cat", Pipeline([
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("onehot", ohe_c)
        ]), cat_c)
    ])

    timing_model = Pipeline([
        ("preprocessor", prep_c),
        ("model", LogisticRegression(
            max_iter=2000,
            class_weight="balanced",
            random_state=RANDOM_STATE
        ))
    ])

    timing_model.fit(Xc_train, yc_train)
    pc = timing_model.predict_proba(Xc_test)[:, 1]
    predc = (pc >= 0.5).astype(int)

    timing_rows.append({
        "Cutoff_Day": cutoff,
        "Accuracy": accuracy_score(yc_test, predc),
        "Recall": recall_score(yc_test, predc, zero_division=0),
        "F1": f1_score(yc_test, predc, zero_division=0),
        "ROC-AUC": roc_auc_score(yc_test, pc),
        "PR-AUC": average_precision_score(yc_test, pc)
    })

timing_results = pd.DataFrame(timing_rows)
display(timing_results.round(4))

plt.figure(figsize=(8, 5))
plt.plot(
    timing_results["Cutoff_Day"],
    timing_results["ROC-AUC"],
    marker="o",
    label="ROC-AUC"
)
plt.plot(
    timing_results["Cutoff_Day"],
    timing_results["PR-AUC"],
    marker="o",
    label="PR-AUC"
)
plt.xlabel("Early-warning cutoff (days)")
plt.ylabel("Score")
plt.title("Performance by Early-Warning Cutoff")
plt.xticks([7, 14, 21, 28])
plt.ylim(0, 1)
plt.grid(True, alpha=0.3)
plt.legend()
plt.tight_layout()
plt.show()


## 49. Bootstrap 95% confidence intervals for the primary model

In [ ]:
def bootstrap_metric_ci(
    y_true,
    probabilities,
    metric_fn,
    n_bootstrap=1000,
    seed=RANDOM_STATE
):
    rng = np.random.RandomState(seed)
    y_arr = np.asarray(y_true)
    p_arr = np.asarray(probabilities)
    scores = []

    for _ in range(n_bootstrap):
        idx = rng.randint(0, len(y_arr), len(y_arr))
        y_sample = y_arr[idx]
        p_sample = p_arr[idx]

        # Metrics such as ROC-AUC/PR-AUC require both classes.
        if len(np.unique(y_sample)) < 2:
            continue

        scores.append(metric_fn(y_sample, p_sample))

    scores = np.asarray(scores)

    return {
        "estimate": metric_fn(y_arr, p_arr),
        "lower_95": np.percentile(scores, 2.5),
        "upper_95": np.percentile(scores, 97.5),
        "n_valid_bootstrap": len(scores)
    }

roc_ci = bootstrap_metric_ci(
    y_test,
    cat_prob,
    roc_auc_score
)

pr_ci = bootstrap_metric_ci(
    y_test,
    cat_prob,
    average_precision_score
)

bootstrap_results = pd.DataFrame([
    {"Metric": "ROC-AUC", **roc_ci},
    {"Metric": "PR-AUC", **pr_ci}
])

display(bootstrap_results.round(4))


## 50. Closed-loop model-update demonstration

This is a **methodological demonstration**, not a claim that OULAD contains intervention-treatment records.

A valid update cycle must train on earlier cohorts and evaluate on a genuinely later cohort. It must never train and test on the same labels.

In [ ]:
available_presentations = sorted(
    data["code_presentation"].dropna().unique()
)

print("Available presentations:", available_presentations)

# Demonstration rule:
# If at least three chronological presentations are available,
# train on the first two and evaluate on the third.
if len(available_presentations) >= 3:
    update_train_presentations = available_presentations[:2]
    update_test_presentations = [available_presentations[2]]

    update_train = data[
        data["code_presentation"].isin(update_train_presentations)
    ].copy()

    update_test = data[
        data["code_presentation"].isin(update_test_presentations)
    ].copy()

    print("Update-cycle training presentations:",
          update_train_presentations)
    print("Update-cycle evaluation presentation:",
          update_test_presentations)
    print("Update training rows:", len(update_train))
    print("Update evaluation rows:", len(update_test))

    print(
        "\nClosed-loop logic:"
        "\n1. Train on earlier cohorts"
        "\n2. Predict current/future cohort"
        "\n3. Observe later outcomes"
        "\n4. Add newly labeled historical cohort"
        "\n5. Retrain"
        "\n6. Evaluate only on a newer unseen cohort"
    )
else:
    print(
        "Not enough chronological presentations for a separate "
        "post-update evaluation. The main temporal experiment remains valid."
    )


## 51. Publication-ready model comparison table

In [ ]:
publication_model_table = results.copy()

publication_model_table[
    ["Accuracy", "Balanced Accuracy", "Precision", "Recall",
     "F1", "ROC-AUC", "PR-AUC", "Brier Score"]
] = publication_model_table[
    ["Accuracy", "Balanced Accuracy", "Precision", "Recall",
     "F1", "ROC-AUC", "PR-AUC", "Brier Score"]
].round(4)

display(publication_model_table)


## 52. Publication-ready ablation table

In [ ]:
display(ablation_results.round(4))


## 53. Publication-ready fairness summary

In [ ]:
fairness_summary_rows = []

for feature_name, table in fairness_tables.items():
    if len(table) == 0:
        continue

    for _, r in table.iterrows():
        fairness_summary_rows.append(r.to_dict())

fairness_summary = pd.DataFrame(fairness_summary_rows)

if len(fairness_summary):
    display(fairness_summary.round(4))
else:
    print("No fairness tables available.")


## 54. Save publication outputs

All result files are written into `publication_outputs/`.

In [ ]:
OUTPUT_DIR = Path("publication_outputs")
OUTPUT_DIR.mkdir(exist_ok=True)

integrity_df.to_csv(
    OUTPUT_DIR / "data_integrity.csv", index=False
)

shape_df.to_csv(
    OUTPUT_DIR / "dataset_shapes.csv", index=False
)

presentation_risk.to_csv(
    OUTPUT_DIR / "presentation_risk.csv", index=False
)

publication_model_table.to_csv(
    OUTPUT_DIR / "model_comparison.csv", index=False
)

threshold_df.to_csv(
    OUTPUT_DIR / "threshold_analysis.csv", index=False
)

risk_predictions.to_csv(
    OUTPUT_DIR / "student_risk_predictions.csv", index=False
)

feature_importance_df.to_csv(
    OUTPUT_DIR / "catboost_feature_importance.csv", index=False
)

ablation_results.to_csv(
    OUTPUT_DIR / "ablation_results.csv", index=False
)

timing_results.to_csv(
    OUTPUT_DIR / "early_warning_timing.csv", index=False
)

bootstrap_results.to_csv(
    OUTPUT_DIR / "bootstrap_confidence_intervals.csv", index=False
)

if len(fairness_summary):
    fairness_summary.to_csv(
        OUTPUT_DIR / "fairness_analysis.csv", index=False
    )

print("Publication outputs saved to:", OUTPUT_DIR.resolve())


## 55. Save SHAP summary data and final model

The saved model is accompanied by the cutoff and cohort information used in the primary experiment.

In [ ]:
# Save a compact SHAP importance table.
try:
    shap_importance = pd.DataFrame({
        "feature": X_shap.columns,
        "mean_abs_shap": np.abs(shap_values.values).mean(axis=0)
    }).sort_values("mean_abs_shap", ascending=False)

    shap_importance.to_csv(
        OUTPUT_DIR / "shap_global_importance.csv",
        index=False
    )

    display(shap_importance.head(25))
except Exception as e:
    print("SHAP table could not be saved:", e)

model_bundle = {
    "model": cat_model,
    "cutoff_day": EARLY_WARNING_DAY,
    "train_presentations": TRAIN_PRESENTATIONS,
    "test_presentations": TEST_PRESENTATIONS,
    "risk_category_rule": {
        "SAFE": "<0.25",
        "WARNING": "0.25-<0.50",
        "AT-RISK": "0.50-<0.75",
        "CRITICAL": ">=0.75"
    },
    "feature_columns": list(cat_X_train.columns)
}

joblib.dump(
    model_bundle,
    OUTPUT_DIR / "final_catboost_28day_model.joblib"
)

print(
    "Final CatBoost model saved to:",
    OUTPUT_DIR / "final_catboost_28day_model.joblib"
)


# 56. Final research summary

The notebook is complete when all cells run successfully and the generated values are used in the manuscript.

### Primary claims supported by this notebook
1. A 28-day early-warning prediction task is defined without using future outcome information.
2. Assessment and LMS features are restricted to information available by the prediction cutoff.
3. Temporal train/test evaluation is used for the main experiment.
4. Multiple models are compared.
5. Risk probabilities and operational risk categories are generated.
6. SHAP and LIME provide global/local explanations.
7. Risk factors are mapped to support recommendations.
8. Day-28/56/84 engagement is retrospectively monitored.
9. Calibration and subgroup fairness diagnostics are reported.
10. Ablation and early-warning timing experiments quantify the contribution of feature sources and prediction timing.
11. Bootstrap confidence intervals quantify uncertainty.
12. A valid closed-loop update methodology is demonstrated without claiming causal intervention effects.

### Publication caution
Do not copy old random-split performance numbers into the manuscript. The final manuscript should report the results generated by the corrected temporal experiment after all seven CSVs are uploaded and the notebook is run successfully.
